# 00 — Shared Data and Final Comparison

**EN3150 Assignment 03 — Resource-Constrained CNN for Edge Image Classification**

**Dataset:** UCI Jute Pest Dataset  
**Task:** 17-class pest image classification  
**Input size:** 64 × 64 × 3  
**Split:** 70% train / 15% validation / 15% test  
**Seed:** 42

## Purpose

This notebook does two jobs:

1. Prepare and verify the **common UCI Jute Pest dataset** for the whole group.
2. After all model training is complete, load the four result JSON files and make the final comparison.

All members must use the same dataset, class order, image size, seed, and split.

In [ ]:
# TensorFlow is already available in Google Colab.
%pip install -q scikit-learn pillow

In [ ]:
import os
import json
import random
import shutil
import zipfile
import urllib.request
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from PIL import Image
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
SEED = 42
IMG_SIZE = 64
BATCH_SIZE = 64

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("SEED =", SEED)
print("IMG_SIZE =", IMG_SIZE)
print("BATCH_SIZE =", BATCH_SIZE)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Sahanya's personal storage
PERSONAL_ROOT = Path(
    "/content/drive/MyDrive/EN3150_A03_PERSONAL/sahanya"
)

DATA_ROOT = PERSONAL_ROOT / "dataset_cache" / "jute_pest"
PLOT_ROOT = PERSONAL_ROOT / "plots"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

# Group-shared storage
SHARED_ROOT = Path(
    "/content/drive/MyDrive/EN3150_A03_SHARED"
)

SHARED_ROOT.mkdir(parents=True, exist_ok=True)

# Use a NEW result directory so old TF Flowers results cannot mix in.
RESULT_ROOT = SHARED_ROOT / "shared_results_jute_pest"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("Dataset cache:", DATA_ROOT)
print("Shared result folder:", RESULT_ROOT)

## 1. UCI Jute Pest dataset

We use the **Jute Pest Dataset** from the UCI Machine Learning Repository.

- UCI dataset ID: **920**
- UCI reports **7,235 images**
- Number of classes: **17**
- Task: image classification
- Images are resized to the assignment maximum of **64 × 64**
- We create our own **stratified 70% / 15% / 15% split with seed 42**

The UCI archive already contains original train/validation/test folders, but the assignment specifically requires a 70/15/15 split. Therefore, this notebook combines all usable images first and re-splits them.

UCI page:  
`https://archive.ics.uci.edu/dataset/920/jute+pest+dataset`

DOI: `10.24432/C5289P`

In [ ]:
# ============================================================
# DOWNLOAD + PREPARE UCI JUTE PEST DATASET
# ============================================================

UCI_DATASET_PAGE = (
    "https://archive.ics.uci.edu/dataset/920/"
    "jute+pest+dataset"
)

DATASET_URL = (
    "https://archive.ics.uci.edu/static/public/920/"
    "jute+pest+dataset.zip"
)

ZIP_PATH = DATA_ROOT / "jute_pest_dataset.zip"
EXTRACT_DIR = DATA_ROOT / "extracted"
EXTRACT_MARKER = EXTRACT_DIR / ".extraction_complete"

# Class order published by UCI.
CLASS_NAMES = [
    "Beet Armyworm",
    "Black Hairy",
    "Cutworm",
    "Field Cricket",
    "Jute Aphid",
    "Jute Hairy",
    "Jute Red Mite",
    "Jute Semilooper",
    "Jute Stem Girdler",
    "Jute Stem Weevil",
    "Leaf Beetle",
    "Mealybug",
    "Pod Borer",
    "Scopula Emissaria",
    "Termite",
    "Termite odontotermes (Rambur)",
    "Yellow Mite",
]

NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES == 17

# ------------------------------------------------------------
# 1. Download once
# ------------------------------------------------------------
if not ZIP_PATH.exists():
    print("Downloading UCI Jute Pest dataset (~155 MB)...")
    print("This is only needed the first time.")

    temp_zip = ZIP_PATH.with_suffix(".part")

    if temp_zip.exists():
        temp_zip.unlink()

    urllib.request.urlretrieve(
        DATASET_URL,
        temp_zip
    )

    temp_zip.replace(ZIP_PATH)
    print("Download complete.")

else:
    print("Using cached ZIP:", ZIP_PATH)

if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(
        "Downloaded file is not a valid ZIP. "
        "Delete it and rerun this cell."
    )

# ------------------------------------------------------------
# 2. Extract once
# ------------------------------------------------------------
if not EXTRACT_MARKER.exists():

    print("Extracting dataset...")

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

    EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zf:
        zf.extractall(EXTRACT_DIR)

    EXTRACT_MARKER.write_text(
        "Extraction complete",
        encoding="utf-8"
    )

    print("Extraction complete.")

else:
    print("Using cached extracted dataset:", EXTRACT_DIR)

# ------------------------------------------------------------
# 3. Robust class-folder matching
# ------------------------------------------------------------
def normalize_name(name):
    name = name.lower()
    name = re.sub(r"[_\\-]+", " ", name)
    name = re.sub(r"[^a-z0-9 ]+", " ", name)
    name = re.sub(r"\\s+", " ", name).strip()
    return name


name_to_label = {
    normalize_name(name): i
    for i, name in enumerate(CLASS_NAMES)
}

# Possible shorter spelling for class 15
name_to_label[
    normalize_name("Termite odontotermes")
] = 15

name_to_label[
    normalize_name("Termite odontotermes Rambur")
] = 15


def infer_label(path):
    # Search parent folders for either:
    # - numeric class folder 0..16
    # - named class folder

    for parent in path.parents:

        if parent == EXTRACT_DIR.parent:
            break

        folder = parent.name.strip()

        if folder.isdigit():
            idx = int(folder)

            if 0 <= idx < NUM_CLASSES:
                return idx

        normalized = normalize_name(folder)

        if normalized in name_to_label:
            return name_to_label[normalized]

    return None

# ------------------------------------------------------------
# 4. Discover and validate images
# ------------------------------------------------------------
VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

candidate_images = sorted(
    p
    for p in EXTRACT_DIR.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower() in VALID_EXTENSIONS
    )
)

print("Image files discovered:", len(candidate_images))

if len(candidate_images) == 0:
    raise RuntimeError(
        "No image files were found after extraction."
    )

relative_paths = []
labels = []

skipped_unlabelled = []
skipped_corrupt = []

for path in candidate_images:

    label = infer_label(path)

    if label is None:
        skipped_unlabelled.append(str(path))
        continue

    try:
        with Image.open(path) as img:
            img.verify()

    except Exception:
        skipped_corrupt.append(str(path))
        continue

    relative_paths.append(
        str(path.relative_to(EXTRACT_DIR))
    )

    labels.append(label)

all_relative_paths = np.asarray(
    relative_paths,
    dtype=str
)

all_labels = np.asarray(
    labels,
    dtype=np.int32
)

print("Usable labelled images:", len(all_relative_paths))
print("Unlabelled skipped:", len(skipped_unlabelled))
print("Corrupt skipped:", len(skipped_corrupt))

if len(all_relative_paths) < 7000:
    print(
        "WARNING: fewer than 7000 usable images were found. "
        "Check the extracted archive before continuing."
    )

assert len(all_relative_paths) == len(all_labels)
assert len(np.unique(all_labels)) == 17

# ------------------------------------------------------------
# 5. Assignment-required STRATIFIED 70/15/15 split
# ------------------------------------------------------------
(
    train_rel_paths,
    temp_rel_paths,
    train_labels,
    temp_labels,
) = train_test_split(
    all_relative_paths,
    all_labels,
    test_size=0.30,
    random_state=SEED,
    stratify=all_labels,
)

(
    val_rel_paths,
    test_rel_paths,
    val_labels,
    test_labels,
) = train_test_split(
    temp_rel_paths,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels,
)

print()
print("Split sizes")
print("-----------")
print("Train:", len(train_rel_paths))
print("Validation:", len(val_rel_paths))
print("Test:", len(test_rel_paths))
print(
    "Total:",
    len(train_rel_paths)
    + len(val_rel_paths)
    + len(test_rel_paths)
)

# ------------------------------------------------------------
# 6. Convert relative paths to absolute paths
# ------------------------------------------------------------
def to_absolute(relative_array):
    return np.asarray(
        [
            str(EXTRACT_DIR / rel_path)
            for rel_path in relative_array
        ],
        dtype=str,
    )


train_paths = to_absolute(train_rel_paths)
val_paths = to_absolute(val_rel_paths)
test_paths = to_absolute(test_rel_paths)

# ------------------------------------------------------------
# 7. Build raw tf.data datasets
# ------------------------------------------------------------
def decode_image(path, label):

    image_bytes = tf.io.read_file(path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )

    image.set_shape(
        [None, None, 3]
    )

    return image, label


raw_train = (
    tf.data.Dataset
    .from_tensor_slices(
        (train_paths, train_labels)
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)

raw_val = (
    tf.data.Dataset
    .from_tensor_slices(
        (val_paths, val_labels)
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)

raw_test = (
    tf.data.Dataset
    .from_tensor_slices(
        (test_paths, test_labels)
    )
    .map(
        decode_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
)


def count_examples(ds):
    return int(
        tf.data.experimental
        .cardinality(ds)
        .numpy()
    )


print()
print("TensorFlow cardinality")
print("----------------------")
print("Train:", count_examples(raw_train))
print("Validation:", count_examples(raw_val))
print("Test:", count_examples(raw_test))

assert count_examples(raw_train) == len(train_paths)
assert count_examples(raw_val) == len(val_paths)
assert count_examples(raw_test) == len(test_paths)

assert set(np.unique(train_labels)) == set(range(17))
assert set(np.unique(val_labels)) == set(range(17))
assert set(np.unique(test_labels)) == set(range(17))

print()
print("UCI Jute Pest dataset loaded successfully.")

In [ ]:
# ============================================================
# RESIZE TO 64 × 64 AND BUILD INPUT PIPELINES
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        antialias=True,
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


# Cache before shuffle so training can reshuffle each epoch.
train_ds = (
    raw_train
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .shuffle(
        4096,
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    raw_val
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    raw_test
    .map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Prepared 64×64 pipelines.")
print(
    "Training batches:",
    tf.data.experimental.cardinality(train_ds).numpy()
)
print(
    "Validation batches:",
    tf.data.experimental.cardinality(val_ds).numpy()
)
print(
    "Test batches:",
    tf.data.experimental.cardinality(test_ds).numpy()
)

In [ ]:
# ============================================================
# DISPLAY EXAMPLES
# ============================================================

plt.figure(figsize=(12, 10))

for images, labels_batch in train_ds.take(1):

    for i in range(
        min(12, len(images))
    ):

        plt.subplot(
            3,
            4,
            i + 1
        )

        plt.imshow(
            tf.cast(
                images[i],
                tf.uint8
            )
        )

        plt.title(
            CLASS_NAMES[
                int(labels_batch[i])
            ],
            fontsize=9
        )

        plt.axis("off")

plt.tight_layout()

plt.savefig(
    PLOT_ROOT
    / "jute_pest_examples_64x64.png",
    dpi=180,
)

plt.show()

In [ ]:
# ============================================================
# SAVE EXACT SPLIT MANIFEST + DATASET SUMMARY
# ============================================================

def make_manifest(
    relative_paths,
    split_labels,
    split_name
):

    return pd.DataFrame({
        "relative_path": relative_paths,
        "label": split_labels.astype(int),
        "class_name": [
            CLASS_NAMES[int(x)]
            for x in split_labels
        ],
        "split": split_name,
    })


manifest = pd.concat(
    [
        make_manifest(
            train_rel_paths,
            train_labels,
            "train"
        ),
        make_manifest(
            val_rel_paths,
            val_labels,
            "validation"
        ),
        make_manifest(
            test_rel_paths,
            test_labels,
            "test"
        ),
    ],
    ignore_index=True,
)

manifest_path = (
    RESULT_ROOT
    / "jute_pest_split_manifest.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)


class_counts = {
    class_name: int(
        np.sum(
            all_labels == class_index
        )
    )
    for class_index, class_name
    in enumerate(CLASS_NAMES)
}


dataset_summary = {
    "dataset": "UCI Jute Pest",
    "uci_dataset_id": 920,
    "uci_dataset_page": UCI_DATASET_PAGE,
    "dataset_download_url": DATASET_URL,
    "doi": "10.24432/C5289P",
    "task": "17-class image classification",
    "image_size": [64, 64, 3],
    "split_method": "stratified random split",
    "split": {
        "train": "70%",
        "validation": "15%",
        "test": "15%",
    },
    "seed": SEED,
    "num_classes": NUM_CLASSES,
    "classes": CLASS_NAMES,
    "usable_images": int(
        len(all_relative_paths)
    ),
    "train_samples": int(
        len(train_rel_paths)
    ),
    "validation_samples": int(
        len(val_rel_paths)
    ),
    "test_samples": int(
        len(test_rel_paths)
    ),
    "skipped_unlabelled": int(
        len(skipped_unlabelled)
    ),
    "skipped_corrupt": int(
        len(skipped_corrupt)
    ),
    "class_counts": class_counts,
}


summary_path = (
    RESULT_ROOT
    / "dataset_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        dataset_summary,
        f,
        indent=2
    )


print(
    json.dumps(
        dataset_summary,
        indent=2
    )
)

print()
print("Manifest:", manifest_path)
print("Summary:", summary_path)

In [ ]:
# ============================================================
# VERIFY CLASS DISTRIBUTION
# ============================================================

distribution = (
    manifest
    .groupby(
        ["class_name", "split"]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(CLASS_NAMES)
)

display(distribution)

distribution.to_csv(
    RESULT_ROOT
    / "jute_pest_class_distribution.csv"
)

ax = distribution.plot(
    kind="bar",
    figsize=(14, 6)
)

ax.set_title(
    "UCI Jute Pest — Stratified 70/15/15 Split"
)

ax.set_xlabel("Pest class")
ax.set_ylabel("Number of images")

plt.xticks(
    rotation=70,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    PLOT_ROOT
    / "jute_pest_class_distribution.png",
    dpi=180,
)

plt.show()

### ✅ Git commit checkpoint

Suggested commit:

```text
feat: add UCI Jute Pest shared data preparation and stratified split
```

Do not commit the 155 MB dataset ZIP or extracted images to GitHub.

# STOP HERE until model training is complete

At this stage the shared folder should contain:

```text
EN3150_A03_SHARED/
└── shared_results_jute_pest/
    ├── dataset_summary.json
    ├── jute_pest_split_manifest.csv
    └── jute_pest_class_distribution.csv
```

The four model notebooks must use this **same manifest/split**.

Later this notebook expects:

```text
model_a.json
model_b.json
mobilenetv2.json
efficientnetb0.json
```

## 2. Final model comparison

Run the cells below **only after all four models have been trained and evaluated on the Jute Pest dataset**.

In [ ]:
required = [
    "model_a.json",
    "model_b.json",
    "mobilenetv2.json",
    "efficientnetb0.json",
]

missing = [
    filename
    for filename in required
    if not (
        RESULT_ROOT / filename
    ).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing Jute Pest result files: "
        + ", ".join(missing)
    )

records = [
    json.load(
        open(
            RESULT_ROOT / filename
        )
    )
    for filename in required
]

comparison = pd.DataFrame(
    records
)

display(comparison)

comparison.to_csv(
    RESULT_ROOT
    / "final_model_comparison.csv",
    index=False,
)

print(
    "Saved:",
    RESULT_ROOT
    / "final_model_comparison.csv"
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    comparison["model_size_mb"],
    comparison["accuracy"],
    s=80,
)

for _, row in comparison.iterrows():

    plt.annotate(
        row["model"],
        (
            row["model_size_mb"],
            row["accuracy"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xlabel("Saved model size (MB)")
plt.ylabel("Test accuracy")
plt.title("Jute Pest — Accuracy vs Model Size")
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    PLOT_ROOT
    / "final_accuracy_vs_model_size.png",
    dpi=180,
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    comparison["parameters"],
    comparison["accuracy"],
    s=80,
)

for _, row in comparison.iterrows():

    plt.annotate(
        row["model"],
        (
            row["parameters"],
            row["accuracy"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.xscale("log")
plt.xlabel("Parameters (log scale)")
plt.ylabel("Test accuracy")
plt.title("Jute Pest — Accuracy vs Parameter Count")
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    PLOT_ROOT
    / "final_accuracy_vs_parameters.png",
    dpi=180,
)

plt.show()

## Report discussion

Use the measured results to discuss:

- Standard CNN Model A vs lightweight Model B,
- accuracy,
- parameter count,
- model size,
- training time per epoch,
- computational cost,
- common class confusions,
- the effect of depthwise-separable convolution,
- Model B vs MobileNetV2 and EfficientNetB0.

Do not decide the conclusion before the measured results are available.

### ✅ Final Git commit checkpoint

Suggested commit:

```text
analysis: add final Jute Pest model comparison
```